# Memory Retrieval Benchmark

This notebook measures memory retrieval latency via two paths:

- Direct API: `POST /api/companions/{id}/memories/search` (end-to-end HTTP time).
- Text route: `POST /conversations/{id}/messages` (parses server `timings` for `retrieval_ms`, `retrieval_embed_ms`, `retrieval_db_ms`).

Notes
- The server exposes `MEMORY_RETRIEVAL_TIMEOUT_MS` in milliseconds (default 600 ms).
- The text route includes retrieval only if memory is enabled for the companion and gating passes.
- The embedding cache (30s TTL by default) can hide embedding costs on repeated queries. Use `vary_query=True` to avoid cache hits.

Prereqs
- Set `EM_API_BASE` (e.g., http://localhost:8100).
- Set `EM_BEARER_TOKEN` to a valid user JWT if your API requires auth for `/api/...` endpoints.
- Set `EM_COMPANION_ID` to a companion you own with memory enabled.

Outputs
- Per-call timings and aggregates (P50/P95) for each scenario.

In [ ]:
import json
import os
import random
import string
import time
from typing import Any, Dict, List

import httpx
import pandas as pd

BASE_URL = os.getenv("EM_API_BASE", "http://localhost:8100").rstrip("/")
COMPANION_ID = "ece1da13-92b3-43f2-bb67-8b1fbc527afa"  # os.getenv('EM_COMPANION_ID', '')
TOKEN = os.getenv("EM_BEARER_TOKEN", "")
TIMEOUT_S = float(os.getenv("EM_HTTP_TIMEOUT", "30"))
HEADERS = {"Content-Type": "application/json", "X-EM-Debug-Timing": "1"}
if TOKEN:
    HEADERS["Authorization"] = f"Bearer {TOKEN}"

assert COMPANION_ID, "Set EM_COMPANION_ID to your companion UUID"
client = httpx.Client(timeout=TIMEOUT_S, follow_redirects=True)


def api(path: str) -> str:
    path = path if path.startswith("/") else "/" + path
    return f"{BASE_URL}{path}"


def rand_suffix(n: int = 6) -> str:
    return "".join(random.choice(string.ascii_lowercase) for _ in range(n))


def create_conversation(companion_id: str) -> str:
    r = client.post(
        api("/conversations/"),
        headers={"Content-Type": "application/json"},
        json={"companion_id": companion_id},
    )
    r.raise_for_status()
    return r.json()["id"]


def send_text(
    conversation_id: str,
    content: str,
    provider: str = "openai-gpt4o-mini",
    temperature: float = 0.1,
) -> Dict[str, Any]:
    body = {
        "content": content,
        "system_prompt": "You are a helpful assistant.",
        "llm_provider": provider,
        "temperature": temperature,
    }
    t0 = time.perf_counter()
    r = client.post(api(f"/conversations/{conversation_id}/messages"), headers=HEADERS, json=body)
    t1 = time.perf_counter()
    r.raise_for_status()
    data = r.json()
    timings = data.get("timings") or {}
    timings = {k: float(v) for k, v in timings.items()} if isinstance(timings, dict) else {}
    timings["total_client_ms"] = (t1 - t0) * 1000.0
    return {"data": data, "timings": timings}


def memory_search(
    companion_id: str,
    query: str,
    *,
    top_k: int = 15,
    min_saliency: float = 0.2,
    external_user_id: str | None = None,
) -> Dict[str, Any]:
    body = {
        "query": query,
        "top_k": top_k,
        "min_saliency": min_saliency,
        "external_user_id": external_user_id,
    }
    t0 = time.perf_counter()
    r = client.post(
        api(f"/api/companions/{companion_id}/memories/search"), headers=HEADERS, json=body
    )
    t1 = time.perf_counter()
    r.raise_for_status()
    data = r.json()
    debug = {}
    dbg_hdr = r.headers.get("X-EM-Debug-Timing")
    if dbg_hdr:
        try:
            debug = json.loads(dbg_hdr)
        except Exception:
            debug = {}
    return {"resp": data, "client_ms": (t1 - t0) * 1000.0, "debug": debug}

## Scenario A — Direct search latency (HTTP end-to-end)
- Uses `/api/companions/{id}/memories/search`.
- Set `vary_query=True` to avoid server-side embedding cache hits.

In [ ]:
N = 5
BASE_QUERY = "favorite coffee preference"
vary_query = True
rows = []
for i in range(N):
    q = f"{BASE_QUERY} {rand_suffix(4)}" if vary_query else BASE_QUERY
    r = memory_search(COMPANION_ID, q, top_k=12, min_saliency=0.2)
    n_items = len((r["resp"] or {}).get("items", []))
    dbg = r.get("debug") or {}
    rows.append(
        {
            "i": i,
            "client_ms": r["client_ms"],
            "items": n_items,
            "embed_ms": dbg.get("retrieval_embed_ms"),
            "db_ms": dbg.get("retrieval_db_ms"),
            "db_knn_ms": dbg.get("db_knn_ms"),
            "db_rerank_ms": dbg.get("db_rerank_ms"),
            "cfg_ms": dbg.get("cfg_ms"),
            "auth_ms": dbg.get("auth_ms"),
            "owner_ms": dbg.get("owner_ms"),
            "items_model_ms": dbg.get("items_model_ms"),
        }
    )
df_search = pd.DataFrame(rows)
df_search.head()

In [ ]:
for i in range(N):
    print(df_search.iloc[i].to_dict())

In [ ]:
def summarize_latency_ms(vals: List[float]) -> Dict[str, float]:
    if not vals:
        return {}
    s = sorted(vals)

    def pct(p):
        k = max(0, min(len(s) - 1, int(round(p * (len(s) - 1)))))
        return s[k]

    return {
        "count": len(s),
        "p50_ms": pct(0.50),
        "p95_ms": pct(0.95),
        "min_ms": s[0],
        "max_ms": s[-1],
        "avg_ms": sum(s) / len(s),
    }


summary_search = summarize_latency_ms(df_search["client_ms"].tolist())
summary_search

## Scenario B — Text route retrieval timings (server-provided breakdown)
- Sends a message that triggers lexical gating (`remember`, `i prefer`, etc.).
- Parses `retrieval_ms`, `retrieval_embed_ms`, and `retrieval_db_ms` from the response `timings`.

In [ ]:
conv = create_conversation(COMPANION_ID)
N = 10
rows = []
for i in range(N):
    # Trigger retrieval via lexical cues; vary slightly to avoid caching if desired
    msg = f"Remember this preference: I prefer filter coffee #{i}"
    r = send_text(conv, msg)
    t = r["timings"]
    rows.append(
        {
            "i": i,
            "total_client_ms": t.get("total_client_ms", float("nan")),
            "retrieval_ms": t.get("retrieval_ms", float("nan")),
            "retrieval_embed_ms": t.get("retrieval_embed_ms", float("nan")),
            "retrieval_db_ms": t.get("retrieval_db_ms", float("nan")),
            "retrieval_items": t.get("retrieval_items", float("nan")),
            "llm_ms": t.get("llm_ms", float("nan")),
        }
    )
df_text = pd.DataFrame(rows)
df_text

In [ ]:
# print df_text.head() as json
for i in range(10):
    print(df_text.iloc[i].to_dict())

In [ ]:
summary_text_total = summarize_latency_ms(df_text["total_client_ms"].dropna().tolist())
summary_embed = summarize_latency_ms(df_text["retrieval_embed_ms"].dropna().tolist())
summary_db = summarize_latency_ms(df_text["retrieval_db_ms"].dropna().tolist())
{"text_total_ms": summary_text_total, "embed_ms": summary_embed, "db_ms": summary_db}

### Tips
- If you consistently see `retrieval_ms` near the timeout (e.g., ~600 ms) and `retrieval_db_ms` missing/NaN, the timeout likely hit before the DB step completed. Increase `MEMORY_RETRIEVAL_TIMEOUT_MS` or ensure embeddings are cached/reused.
- To isolate DB latency, run Scenario B twice with identical messages; the second run should hit the embedding cache and reveal `retrieval_db_ms` alone.
- To stress test, increase `N` and use `vary_query=True` in Scenario A.